<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 03 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">广告数据标准化和合并</div>
  <p class="doris-cover-lead">加载 Google、Meta 和 TikTok CSV，统一字段、删除重复记录并合并为一张广告明细表。</p>
  <span class="doris-cover-note">Seed · dbt_utils · View · QUALIFY · Table · Data Test</span>
</div>

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 dbt-for-apache-doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 3：广告数据标准化和合并

这个 Demo 展示 CSV 如何通过 dbt Seed 进入 Doris，再经过字段标准化、`QUALIFY` 去重和 `union all` 合并。

<div class="doris-flow">
  <div class="doris-flow-step"><strong>CSV 输入</strong>Google、Meta、TikTok 三个文件</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>dbt Seed</strong>加载为 Doris Table</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>标准化去重</strong>统一字段并使用 <code>QUALIFY</code></div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>合并明细</strong><code>int__ads_unified</code></div>
</div>

### 2.1 准备 CSV 并执行 dbt Seed

Google 和 Meta 的 8 月 1 日有重复行，TikTok 没有重复行。先由 dbt seed 将三个 CSV 加载成 Doris 表。

In [ ]:
ads_dir = runner.examples_root / "doris-demos/consolidate"
runner.show_file("Fixture SQL", ads_dir / "scripts/setup.sql")
runner.show_file("Google CSV", ads_dir / "seeds/googleads.csv")
runner.show_file("Meta CSV", ads_dir / "seeds/metaads.csv")
runner.show_file("TikTok CSV", ads_dir / "seeds/tiktokads.csv")
runner.run_sql_file("创建广告 Demo 数据库", ads_dir / "scripts/setup.sql")
runner.run_dbt("安装 dbt_utils", ads_dir, "deps")
runner.run_dbt("加载三个广告 Seed", ads_dir, "seed", "--select", "googleads", "metaads", "tiktokads")
runner.query("Seed 输入行数", """
select 'googleads' as table_name, count(*) as table_rows from dbt_demo_consolidate.googleads
union all select 'metaads', count(*) from dbt_demo_consolidate.metaads
union all select 'tiktokads', count(*) from dbt_demo_consolidate.tiktokads
order by table_name
""")

### 2.2 标准化字段并去重

三个 staging Model 使用 `ref()` 读取 Seed：Meta 将 `views_1 + views_2` 合成 `views`，TikTok 将 `views_1` 映射为 `views`，然后用 `row_number()` + `QUALIFY` 删除重复记录。

In [ ]:
for model_name in ("googleads", "metaads", "tiktokads"):
    runner.show_file(f"{model_name} staging Model", ads_dir / f"models/stg__ads_{model_name}.sql")
runner.run_dbt("创建三个 staging View", ads_dir, "run", "--select", "stg__ads_googleads", "stg__ads_metaads", "stg__ads_tiktokads")
runner.query("中间结果：去重后的 staging", """
select 'google' as source, count(*) as rows_after_dedup from dbt_demo_consolidate.stg__ads_googleads
union all select 'meta', count(*) from dbt_demo_consolidate.stg__ads_metaads
union all select 'tiktok', count(*) from dbt_demo_consolidate.stg__ads_tiktokads
order by source
""")

### 2.3 合并三个渠道并执行唯一性测试

最终 Model 给每个 staging 增加 `source` 字段，用 `union all` 合并为一张表；`dbt_utils.unique_combination_of_columns` 检查 `source + ad_date` 不重复。

In [ ]:
runner.show_file("统一广告 Model", ads_dir / "models/int__ads_unified.sql")
runner.show_file("唯一性 Test 定义", ads_dir / "models/int__ads_unified.yml")
runner.run_dbt("创建统一广告表并测试", ads_dir, "build", "--select", "int__ads_unified")
runner.query("输出：统一广告明细", """
select source, ad_date, clicks, impressions, views, conversions
from dbt_demo_consolidate.int__ads_unified
order by source, ad_date
""")

### 2.4 验证最终对象

Verifier 检查最终表有 6 行、日期非空，且三个 staging 对象是 View、统一对象是 Table。

In [ ]:
runner.run_script("校验广告合并 Demo", ads_dir / "scripts/verify.sh")
runner.query("最终对象类型", """
select table_name, table_type
from information_schema.tables
where table_schema = 'dbt_demo_consolidate'
  and table_name in ('stg__ads_googleads', 'stg__ads_metaads', 'stg__ads_tiktokads', 'int__ads_unified')
order by table_name
""")

## 完成

三个 Seed、三个标准化 View、统一广告 Table 和唯一性测试均已通过校验。